# 📓 Notebook 23 — Deep Agents---**Phase 8 · Deep Agent Patterns**> This notebook covers the **cutting-edge** agent architectures used in production today. These are the patterns behind systems like Devin, OpenAI's Operator, Google's Project Mariner, and enterprise-grade autonomous agents.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Agentic RAG | Agents that decide WHEN and HOW to retrieve || Autonomous agent loops | Self-directed agents with goal tracking || Supervisor & swarm patterns | Advanced multi-agent orchestration || MCP (Model Context Protocol) | Standardized tool/data protocol by Anthropic || A2A (Agent-to-Agent) | Google's protocol for inter-agent communication || Agent evaluation | How to benchmark agent performance || Agent-as-a-Service | Deploying agents as APIs |### 🏗️ Build: Autonomous Research Agent with Goal Tracking

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Agentic RAG — When the Agent Controls Retrieval### Standard RAG vs Agentic RAG| Aspect | Standard RAG | Agentic RAG ||--------|-------------|-------------|| **Retrieval** | Always retrieves | Agent decides IF and WHEN to retrieve || **Query** | User's question as-is | Agent reformulates/decomposes queries || **Sources** | One vector store | Agent selects from multiple sources || **Multi-hop** | Single retrieval | Chain of retrieval-reasoning steps || **Routing** | Fixed pipeline | Dynamic routing based on query type |```Standard RAG:   Query → Retrieve → Generate → AnswerAgentic RAG:    Query → Think → (Retrieve? Rewrite? Skip?) → Think → ... → Answer```> **Interview Critical:** Agentic RAG is the future of RAG. The agent becomes the **orchestrator** of retrieval, not just a consumer of it.

In [ ]:
class AgenticRAG:    """An agent that dynamically decides how to handle retrieval."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        # Simulated knowledge bases        self.knowledge_bases = {            "technical": [                "Python is a high-level language created in 1991 by Guido van Rossum.",                "Transformers use self-attention to process sequences in parallel.",                "RAG grounds LLM responses in external knowledge to reduce hallucination.",                "Vector databases store and search high-dimensional embeddings efficiently.",                "FAISS is a library for efficient similarity search by Facebook AI Research.",            ],            "business": [                "The AI market is expected to reach $1.8 trillion by 2030.",                "Enterprise AI adoption grew by 270% between 2020 and 2024.",                "Top AI spending sectors: healthcare, finance, retail, manufacturing.",                "ROI of AI projects averages 3.5x in the first year of deployment.",            ],            "current_events": [                "GPT-4o was released in May 2024 with multimodal capabilities.",                "Claude 3.5 Sonnet became the top coding model in late 2024.",                "Google Gemini 2.0 introduced native tool use and agentic features.",                "Open-source models like Llama 3 now rival proprietary models in benchmarks.",            ],        }        def _classify_query(self, query):        """Decide: Answer from knowledge? Retrieve? Which source?"""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"""Classify this query. Respond in JSON:{{    "needs_retrieval": true/false,    "source": "technical" or "business" or "current_events" or "general",    "should_decompose": true/false,    "sub_queries": ["..."] (if should_decompose)}}Query: {query}"""            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)        def _retrieve(self, query, source):        """Simple similarity-based retrieval from selected source."""        if source not in self.knowledge_bases:            return []        # In production: use vector search        docs = self.knowledge_bases[source]        query_lower = query.lower()        scored = [(d, sum(1 for w in query_lower.split() if w in d.lower())) for d in docs]        scored.sort(key=lambda x: x[1], reverse=True)        return [d for d, s in scored[:3] if s > 0]        def query(self, question, verbose=True):        """Agentic RAG pipeline."""        if verbose:            print(f"❓ {question}")                # Step 1: Classify        classification = self._classify_query(question)        if verbose:            print(f"  🧠 Classification: {json.dumps(classification)}")                context = ""                if classification.get("needs_retrieval"):            # Step 2: Decompose if needed            queries = classification.get("sub_queries", [question])            if not queries:                queries = [question]                        # Step 3: Retrieve from appropriate sources            all_docs = []            for q in queries:                source = classification.get("source", "technical")                docs = self._retrieve(q, source)                all_docs.extend(docs)                if verbose:                    print(f"  📚 Retrieved {len(docs)} docs from '{source}' for: {q[:50]}...")                        context = "\n".join(set(all_docs))        else:            if verbose:                print(f"  💡 No retrieval needed — answering from knowledge")                # Step 4: Generate        messages = [{"role": "system", "content": f"Answer the question.{f' Use this context: {context}' if context else ''}"}]        messages.append({"role": "user", "content": question})                response = litellm.completion(model=self.model, messages=messages, temperature=0, max_tokens=400)        answer = response.choices[0].message.content                if verbose:            print(f"  📝 {answer[:150]}...")                return answer# Test Agentic RAGarag = AgenticRAG()print("🤖 Agentic RAG Demo")print("=" * 60)for q in [    "What is 2 + 2?",                               # No retrieval needed    "How does FAISS work for similarity search?",     # Technical retrieval    "What's the ROI of enterprise AI projects?",      # Business retrieval    "What new models were released in 2024?",         # Current events]:    arag.query(q)    print("-" * 40)

---## 3. Autonomous Agent Loop### The Autonomous Agent PatternAn agent that pursues a **goal** through multiple self-directed cycles until completion.```┌────────────────────────────────────────┐│            AUTONOMOUS LOOP              ││                                        ││  Goal → Plan → Execute → Evaluate     ││                    ↑         │         ││                    └────── Not done     ││                              │         ││                           Done → EXIT   │└────────────────────────────────────────┘```> **Interview Tip:** This is the pattern behind AutoGPT, BabyAGI, Devin, and similar systems. The key challenge is **preventing runaway execution** — the agent must have clear stopping conditions.

In [ ]:
class AutonomousAgent:    """A goal-directed autonomous agent with progress tracking."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        self.plan = []        self.completed_steps = []        self.scratchpad = []        self.max_iterations = 10        def _create_plan(self, goal):        """Generate a plan to achieve the goal."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Create a step-by-step plan to achieve this goal. Return JSON with 'steps' array of strings (3-5 steps).\n\nGoal: {goal}"            }],            response_format={"type": "json_object"},            temperature=0.3,        )        data = json.loads(response.choices[0].message.content)        return data.get("steps", [goal])        def _execute_step(self, step, context):        """Execute one step of the plan."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "system",                "content": "Execute the given task thoroughly. Provide detailed, actionable output."            }, {                "role": "user",                "content": f"Task: {step}\n\nContext from previous steps:\n{context if context else 'None'}"            }],            temperature=0.4, max_tokens=500,        )        return response.choices[0].message.content        def _evaluate_progress(self, goal, completed):        """Assess: are we done? Do we need to re-plan?"""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"""Goal: {goal}Completed steps and results:{completed}Evaluate progress. Return JSON:{{    "goal_achieved": true/false,    "progress_pct": 0-100,    "remaining_work": "description of what's left",    "should_replan": true/false}}"""            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)        def run(self, goal, verbose=True):        """Run the autonomous loop."""        if verbose:            print(f"🎯 Goal: {goal}")            print("=" * 60)                # Create initial plan        self.plan = self._create_plan(goal)        if verbose:            print(f"📋 Plan ({len(self.plan)} steps):")            for i, s in enumerate(self.plan, 1):                print(f"   {i}. {s}")                context_parts = []                for iteration in range(self.max_iterations):            if not self.plan:                break                        # Execute next step            step = self.plan.pop(0)            if verbose:                print(f"\n🔄 Iteration {iteration + 1}: {step}")                        result = self._execute_step(step, "\n".join(context_parts[-3:]))            self.completed_steps.append({"step": step, "result": result})            context_parts.append(f"[{step}]: {result[:200]}")                        if verbose:                print(f"   ✅ {result[:100]}...")                        # Evaluate progress every 2 steps            if len(self.completed_steps) % 2 == 0 or not self.plan:                completed_summary = "\n".join([                    f"- {s['step']}: {s['result'][:100]}" for s in self.completed_steps                ])                eval_result = self._evaluate_progress(goal, completed_summary)                                if verbose:                    print(f"\n   📊 Progress: {eval_result.get('progress_pct', '?')}%")                                if eval_result.get("goal_achieved"):                    if verbose:                        print(f"   🎉 Goal achieved!")                    break                # Final synthesis        all_work = "\n".join([f"[{s['step']}]: {s['result']}" for s in self.completed_steps])        final = litellm.completion(            model=self.model,            messages=[{"role": "user", "content": f"Goal: {goal}\n\nAll work done:\n{all_work}\n\nProvide the final, synthesized result."}],            temperature=0.3, max_tokens=600,        ).choices[0].message.content                if verbose:            print(f"\n📝 Final Result:\n{final[:400]}...")                return {            "goal": goal,            "steps_completed": len(self.completed_steps),            "iterations": iteration + 1,            "result": final,        }agent = AutonomousAgent()result = agent.run("Analyze the pros and cons of using LangGraph vs CrewAI for building a customer support agent system")

---## 4. Supervisor & Swarm Patterns### Advanced Multi-Agent Architectures```SUPERVISOR PATTERN              SWARM PATTERN┌──────────────┐               ┌────┐ ┌────┐ ┌────┐│  Supervisor  │               │ A1 │↔│ A2 │↔│ A3 ││  (decides)   │               └──┬─┘ └──┬─┘ └──┬─┘└──┬───┬───┬───┘                  │      │      │   ↓   ↓   ↓                    ↕      ↕      ↕┌──┐ ┌──┐ ┌──┐               ┌────┐ ┌────┐ ┌────┐│W1│ │W2│ │W3│               │ A4 │↔│ A5 │↔│ A6 │└──┘ └──┘ └──┘               └────┘ └────┘ └────┘• Central coordinator          • No central authority• Clear accountability         • Agents hand off dynamically• Single point of failure      • Resilient, but harder to debug```> **Interview Tip:** OpenAI's Swarm framework uses the swarm pattern — agents "hand off" to each other dynamically based on context. No central supervisor.

In [ ]:
class SupervisorAgent:    """Supervisor that dynamically routes tasks to specialist agents."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        self.specialists = {            "coder": "You are an expert Python programmer. Write clean, efficient code with type hints.",            "analyst": "You are a data analyst. Provide quantitative insights with specific numbers.",            "writer": "You are a technical writer. Write clear, well-structured documentation.",            "debugger": "You are a debugging expert. Analyze errors methodically and find root causes.",        }        def _decide_routing(self, task):        """Supervisor decides which specialist handles the task."""        specialists_desc = "\n".join([f"- {k}: {v[:60]}" for k, v in self.specialists.items()])                response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Which specialist should handle this task? Available:\n{specialists_desc}\n\nTask: {task}\n\nRespond with JSON: {{"specialist": "name", "subtasks": ["task1", "task2"] or null}}"            }],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)        def _run_specialist(self, specialist_name, task):        """Execute a task with a specialist agent."""        system = self.specialists.get(specialist_name, self.specialists["writer"])        response = litellm.completion(            model=self.model,            messages=[                {"role": "system", "content": system},                {"role": "user", "content": task}            ],            temperature=0.3, max_tokens=500,        )        return response.choices[0].message.content        def handle(self, task, verbose=True):        """Supervisor routes and coordinates."""        if verbose:            print(f"📋 Task: {task}")                routing = self._decide_routing(task)        specialist = routing.get("specialist", "writer")        subtasks = routing.get("subtasks")                if verbose:            print(f"  🔀 Routed to: {specialist}")                if subtasks:            results = []            for sub in subtasks:                if verbose:                    print(f"  🔧 Subtask: {sub[:50]}...")                result = self._run_specialist(specialist, sub)                results.append(result)            return "\n\n".join(results)        else:            return self._run_specialist(specialist, task)supervisor = SupervisorAgent()for task in [    "Write a Python function to validate email addresses",    "Analyze the performance implications of using Vector vs List in Python",    "Write documentation for a REST API endpoint that creates users",]:    print(f"\n{'=' * 50}")    result = supervisor.handle(task)    print(f"  📝 {result[:100]}...")

In [ ]:
class SwarmAgent:    """Swarm pattern — agents dynamically hand off tasks to each other."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        self.agents = {}        self.handoff_log = []        def register(self, name, system_prompt, can_handoff_to):        self.agents[name] = {            "system_prompt": system_prompt,            "can_handoff_to": can_handoff_to,        }        def run(self, initial_agent, task, max_handoffs=5, verbose=True):        """Execute with dynamic handoffs."""        current = initial_agent        context = task                for i in range(max_handoffs):            agent = self.agents[current]                        # Ask agent to handle or handoff            handoff_options = ", ".join(agent["can_handoff_to"]) if agent["can_handoff_to"] else "none"                        response = litellm.completion(                model=self.model,                messages=[                    {"role": "system", "content": f"{agent['system_prompt']}\n\nYou can hand off to: [{handoff_options}]. If you can fully handle this, respond normally. If another agent is better suited, start your response with HANDOFF:agent_name and explain why."},                    {"role": "user", "content": context}                ],                temperature=0.3, max_tokens=400,            )                        reply = response.choices[0].message.content                        # Check for handoff            if reply.startswith("HANDOFF:"):                next_agent = reply.split(":")[1].split("\n")[0].strip().split()[0]                reason = reply[reply.index("\n"):].strip() if "\n" in reply else "Specialist needed"                                if next_agent in self.agents:                    self.handoff_log.append({"from": current, "to": next_agent, "reason": reason[:80]})                    if verbose:                        print(f"  🔄 {current} → {next_agent}: {reason[:60]}...")                    current = next_agent                    context = f"Previous agent ({self.handoff_log[-1]['from']}) handed this to you.\nOriginal task: {task}\nContext: {reason}"                    continue                        # No handoff — this agent handled it            if verbose:                print(f"  ✅ Handled by: {current}")                print(f"  📝 {reply[:100]}...")                        return {"agent": current, "response": reply, "handoffs": len(self.handoff_log)}                return {"agent": current, "response": "Max handoffs reached", "handoffs": max_handoffs}# Build a swarmswarm = SwarmAgent()swarm.register("triage", "You are a triage agent. Classify incoming requests and hand off to the right specialist.", ["coder", "support", "sales"])swarm.register("coder", "You are a coding expert. Write and debug Python code.", ["support"])swarm.register("support", "You are a customer support agent. Help users with product issues.", ["coder", "sales"])swarm.register("sales", "You are a sales agent. Help with pricing and plans.", ["support"])print("🐝 Swarm Agent Demo")print("=" * 50)for task in [    "How do I fix this Python error: IndexError: list index out of range?",    "I need to upgrade my subscription plan",    "My API calls are returning 500 errors",]:    print(f"\n❓ {task}")    result = swarm.run("triage", task)    print(f"   Handoffs: {result['handoffs']}")

---## 5. MCP (Model Context Protocol) & A2A (Agent-to-Agent)### The Emerging Standards| Protocol | By | Purpose | Analogy ||----------|-----|---------|---------|| **MCP** | Anthropic | Standardized tool/data connection for LLMs | USB-C for AI tools || **A2A** | Google | Inter-agent communication protocol | HTTP for agents |### MCP (Model Context Protocol)```┌─────────────┐     MCP      ┌──────────────────┐│   LLM App   │─────────────→│   MCP Server     ││ (MCP Client)│←─────────────│  (Tool Provider) │└─────────────┘              └──────────────────┘                              • Database access                              • File system                              • API integrations                              • Web search```MCP standardizes HOW LLMs connect to tools and data:- **Resources** — Expose data (like GET endpoints)- **Tools** — Expose actions (like POST endpoints)- **Prompts** — Reusable prompt templates- **Sampling** — Request LLM completions from the server side### A2A (Agent-to-Agent Protocol)```┌─────────┐     A2A       ┌─────────┐│ Agent A  │─────────────→│ Agent B  ││ (Client) │←─────────────│ (Server) │└─────────┘               └─────────┘   • Agent Card (capabilities)   • Task lifecycle (submit → working → done)   • Message passing   • Artifact exchange```A2A standardizes HOW agents talk to each other:- **Agent Cards** — JSON describing agent capabilities (like a résumé)- **Tasks** — Standard lifecycle for work requests- **Message parts** — Text, files, structured data- **Streaming** — Real-time updates via SSE> **Interview Critical:** MCP is becoming the standard for tool integration (adopted by many AI IDEs and platforms). A2A is emerging for multi-agent interop. Know both.

In [ ]:
# Simulating MCP-style tool server patternclass MCPToolServer:    """Simulates MCP server — exposes tools and resources via standard interface."""        def __init__(self, name, description):        self.name = name        self.description = description        self.tools = {}        self.resources = {}        def register_tool(self, name, description, handler, params):        self.tools[name] = {"description": description, "handler": handler, "parameters": params}        def register_resource(self, uri, description, handler):        self.resources[uri] = {"description": description, "handler": handler}        def list_tools(self):        return [{"name": n, "description": t["description"], "parameters": t["parameters"]} for n, t in self.tools.items()]        def call_tool(self, name, args):        if name not in self.tools:            return {"error": f"Tool '{name}' not found"}        return self.tools[name]["handler"](**args)        def read_resource(self, uri):        if uri not in self.resources:            return {"error": f"Resource '{uri}' not found"}        return self.resources[uri]["handler"]()        def get_manifest(self):        return {            "name": self.name,            "description": self.description,            "tools": self.list_tools(),            "resources": list(self.resources.keys()),        }# Create an MCP-style serverdb_server = MCPToolServer("database-server", "Access to application database")db_server.register_tool(    "query_users", "Look up user information by name or ID",    handler=lambda name=None, user_id=None: {"users": [{"id": 1, "name": name or "demo", "plan": "pro"}]},    params={"name": "string", "user_id": "integer"})db_server.register_tool(    "count_records", "Count records in a table",    handler=lambda table: {"table": table, "count": 1523},    params={"table": "string"})db_server.register_resource(    "db://schema", "Database schema information",    handler=lambda: {"tables": ["users", "orders", "products"], "total_records": 50000})# MCP client (the LLM app) discovers and uses toolsprint("🔌 MCP Tool Server Demo")print("=" * 50)manifest = db_server.get_manifest()print(f"Server: {manifest['name']}")print(f"Description: {manifest['description']}")print(f"Tools: {[t['name'] for t in manifest['tools']]}")print(f"Resources: {manifest['resources']}")print(f"\n📊 Query result: {db_server.call_tool('query_users', {'name': 'Abhishek'})}")print(f"📊 Count result: {db_server.call_tool('count_records', {'table': 'users'})}")print(f"📊 Schema: {db_server.read_resource('db://schema')}")

In [ ]:
# Simulating A2A Agent Card patternclass A2AAgent:    """Agent with A2A-style agent card and task handling."""        def __init__(self, name, description, skills, model=None):        self.name = name        self.description = description        self.skills = skills        self.model = model or DEFAULT_MODEL        self.task_queue = []        def get_agent_card(self):        """A2A Agent Card — describes capabilities (like a résumé)."""        return {            "name": self.name,            "description": self.description,            "skills": self.skills,            "url": f"https://agents.example.com/{self.name.lower().replace(' ','-')}",            "version": "1.0.0",            "capabilities": {                "streaming": True,                "pushNotifications": False,            }        }        def submit_task(self, task_description, from_agent=None):        """Accept a task (A2A task lifecycle: submitted → working → done)."""        task = {            "id": f"task_{len(self.task_queue)+1}",            "status": "submitted",            "description": task_description,            "from_agent": from_agent,            "submitted_at": datetime.now().isoformat(),        }        self.task_queue.append(task)                # Execute        task["status"] = "working"        response = litellm.completion(            model=self.model,            messages=[                {"role": "system", "content": self.description},                {"role": "user", "content": task_description}            ],            temperature=0.3, max_tokens=300,        )        task["result"] = response.choices[0].message.content        task["status"] = "done"        task["completed_at"] = datetime.now().isoformat()                return task# Create A2A agentsresearch_agent = A2AAgent(    "Research Agent", "Expert at finding and analyzing information on any topic",    skills=["web_search", "data_analysis", "report_writing"])code_agent = A2AAgent(    "Code Agent", "Expert Python developer who writes clean, tested code",    skills=["python", "code_generation", "debugging", "testing"])# Discover agents (via Agent Cards)print("🤝 A2A Protocol Demo")print("=" * 50)for agent in [research_agent, code_agent]:    card = agent.get_agent_card()    print(f"\n📇 Agent Card: {card['name']}")    print(f"   Skills: {card['skills']}")    print(f"   Endpoint: {card['url']}")# Agent-to-agent task delegationprint(f"\n🔄 Agent-to-Agent Task:")task = code_agent.submit_task(    "Write a Python function that validates credit card numbers using the Luhn algorithm",    from_agent="Research Agent")print(f"   Status: {task['status']}")print(f"   Result: {task['result'][:150]}...")

---## 6. Agent Evaluation & Benchmarks### How to Measure Agent Quality| Metric | What It Measures | How ||--------|-----------------|-----|| **Task completion** | Did the agent finish the task correctly? | Binary or rubric scoring || **Tool accuracy** | Did it call the right tools with right args? | Compare to ground truth || **Reasoning quality** | Was the intermediate thinking sound? | LLM-as-judge on trace || **Efficiency** | How many steps/tokens to complete? | Count steps, measure tokens || **Safety** | Did it refuse unsafe requests? | Adversarial test suite || **Reliability** | Does it work consistently? | Run N times, measure variance |> **Interview Tip:** Agent evaluation is MUCH harder than RAG evaluation. There's no single metric — you need a framework that measures correctness, efficiency, and safety together.

In [ ]:
class AgentEvaluator:    """Evaluate agent performance across multiple dimensions."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        def evaluate_task(self, task, expected_outcome, actual_outcome, trace=None):        """Multi-dimensional agent evaluation."""        eval_prompt = f"""Evaluate this agent's performance:Task: {task}Expected outcome: {expected_outcome}Actual outcome: {actual_outcome}Agent trace: {json.dumps(trace) if trace else 'Not provided'}Rate each dimension 0.0-1.0 and provide brief reasoning. Return JSON:{{    "task_completion": {{"score": <float>, "reason": "..."}},    "accuracy": {{"score": <float>, "reason": "..."}},    "efficiency": {{"score": <float>, "reason": "..."}},    "reasoning_quality": {{"score": <float>, "reason": "..."}},    "overall_score": <float>}}"""                response = litellm.completion(            model=self.model,            messages=[{"role": "user", "content": eval_prompt}],            response_format={"type": "json_object"},            temperature=0,        )        return json.loads(response.choices[0].message.content)evaluator = AgentEvaluator()# Evaluate an agent's outputtest_cases = [    {        "task": "Calculate the compound interest on $10,000 at 5% for 3 years",        "expected": "Compound interest formula: A = P(1+r)^t = 10000(1.05)^3 = $11,576.25. Interest earned: $1,576.25",        "actual": "The compound interest on $10,000 at 5% annual rate for 3 years is $1,576.25, making the total $11,576.25.",    },    {        "task": "Summarize the benefits of RAG in 3 bullet points",        "expected": "1. Reduces hallucination 2. Provides up-to-date info 3. Citable sources",        "actual": "RAG is good for AI applications and helps with information retrieval.",    },]print("📊 Agent Evaluation Demo")print("=" * 60)for tc in test_cases:    result = evaluator.evaluate_task(tc["task"], tc["expected"], tc["actual"])    print(f"\n📋 Task: {tc['task'][:50]}...")    print(f"   Overall: {result['overall_score']}")    for dim in ["task_completion", "accuracy", "efficiency", "reasoning_quality"]:        if dim in result:            d = result[dim]            print(f"   {dim}: {d['score']} — {d['reason'][:60]}")

---## 📝 Interview Quiz — Deep Agents### Q1: What is Agentic RAG and how does it differ from standard RAG?<details><summary>💡 Answer</summary>**Standard RAG:** Fixed pipeline — always retrieves, always from the same source, always the same way.**Agentic RAG:** The agent controls the retrieval process:1. **Decides IF** retrieval is needed (some questions don't need it)2. **Decides WHERE** to retrieve from (multiple knowledge bases)3. **Reformulates** the query for better retrieval4. **Multi-hop** — chains multiple retrieval-reasoning steps5. **Evaluates** retrieval quality and retries if poorThis is the direction all major RAG systems are heading.</details>### Q2: Compare Supervisor and Swarm multi-agent patterns.<details><summary>💡 Answer</summary>| Aspect | Supervisor | Swarm ||--------|-----------|-------|| Control | Centralized | Decentralized || Routing | Supervisor decides | Agents hand off || Failure | Single point of failure | Resilient || Debugging | Easy (follow the supervisor) | Hard (trace handoffs) || Scale | Limited by supervisor | Scales horizontally || Predictability | High | Lower |**Supervisor:** LangGraph, CrewAI (hierarchical process)**Swarm:** OpenAI Swarm, dynamic handoff systems</details>### Q3: What is MCP and why does it matter?<details><summary>💡 Answer</summary>**MCP (Model Context Protocol)** by Anthropic standardizes how LLM apps connect to external tools and data.**Before MCP:** Every tool needs custom integration code — N models × M tools = N×M integrations.**With MCP:** Standard protocol — any MCP client works with any MCP server.Like USB-C unified charging cables, MCP unifies tool connections.**Components:** Resources (read data), Tools (execute actions), Prompts (templates), Sampling (server-side LLM calls).**Adopted by:** Claude Desktop, Cursor, Windsurf, many AI IDEs and platforms.</details>### Q4: How would you evaluate an autonomous agent in production?<details><summary>💡 Answer</summary>Multi-dimensional evaluation:1. **Task completion rate** — Does it achieve the goal? (Binary + rubric)2. **Step efficiency** — How many LLM calls/tool calls to complete?3. **Cost per task** — Total tokens × price4. **Latency** — End-to-end time5. **Safety** — Does it refuse unsafe requests? Does it stay within bounds?6. **Reliability** — Run same task 10x — how consistent are results?7. **Error recovery** — When a step fails, does it recover?**Benchmarks:** SWE-Bench (coding), GAIA (general), AgentBench (Chinese), WebArena (web).</details>### Q5: What is A2A? How does it differ from MCP?<details><summary>💡 Answer</summary>| | MCP | A2A ||-|-----|-----|| **Purpose** | LLM ↔ Tools/Data | Agent ↔ Agent || **Analogy** | USB-C (connect peripherals) | HTTP (communicate between services) || **Who talks** | App to tool server | Agent to agent || **Discovery** | Server manifest | Agent Cards || **Lifecycle** | Request/response | Task (submit → working → done) |**MCP** = How an agent uses tools. **A2A** = How agents collaborate.They're complementary — a production system uses both.</details>

---## ✅ Notebook 23 Summary| Concept | Key Takeaway ||---------|-------------|| Agentic RAG | Agent controls IF, WHERE, and HOW to retrieve || Autonomous agents | Goal-directed loops with progress tracking || Supervisor pattern | Central coordinator routes to specialists || Swarm pattern | Agents dynamically hand off to each other || MCP | Standard protocol for LLM ↔ tool connections || A2A | Standard protocol for agent ↔ agent communication || Agent evaluation | Multi-dimensional: completion, accuracy, efficiency, safety |### 🏁 This is the deepest level — now apply everything in the [Capstone Projects](./capstone/)!